# 03 · Download Airtribune competitions to Google Drive

Used with the permission of the Airtribune administrators.

Downloads every past paragliding event on Airtribune (2012 to today) straight into Google Drive, into `airtribune_igc/<event_id>/`:

- `results.html`: the event's results page
- `<task>.zip`: the task's track archive (the scored IGC package when there is one, otherwise the live-tracking archive)
- `<task>_results.html`: the task's results page
- `airtribune_catalog.csv`, one row per task, in the `airtribune_igc` folder

Newest events first. It can run at the same time as notebooks 01 and 02 (different servers). One request at a time, with a pause between requests.

**If Colab disconnects, just Run All again**: files already in Drive are skipped.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
SUBFOLDER = "airtribune_igc"   # created inside that folder
PAUSE = 2.0          # seconds between requests
MIN_FREE_GB = 2.0    # stop when Drive has less free space than this
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, find the project folder and get the code

In [ ]:
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

drive.mount("/content/drive")
auth.authenticate_user()
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
assert os.path.isdir(ROOT), f"Drive folder not found at {ROOT}"
OUT = os.path.join(ROOT, SUBFOLDER)
os.makedirs(OUT, exist_ok=True)
print("Saving to", OUT)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)

usage = shutil.disk_usage(OUT)
print(f"Drive: {usage.free / 1e9:.0f} GB free of {usage.total / 1e9:.0f} GB")

## 3. Catalog and download

In [ ]:
proc = subprocess.Popen(
    ["python3", "-u", f"{REPO}/scripts/airtribune.py", OUT, "--pause", str(PAUSE), "--min-free-gb", str(MIN_FREE_GB)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

## 4. Check what is in Drive

In [ ]:
import zipfile
from pathlib import Path

zips = sorted(Path(OUT).glob("*/*.zip"))
tracks = 0
for z in zips:
    try:
        tracks += sum(name.lower().endswith(".igc") for name in zipfile.ZipFile(z).namelist())
    except zipfile.BadZipFile:
        print("bad zip:", z)
print(f"{len(zips)} task packages, {tracks} IGC tracks, {sum(z.stat().st_size for z in zips) / 1e9:.1f} GB")